### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="customer_satisfaction_in_airline",
    dataset_year="2023",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/yakhyojon/customer-satisfaction-in-airline",
    download_description="""
We download the data from Kaggle and uzip it to a predefined folder.

mkdir -p local-data-warehouse/customer_satisfaction_in_airline/ && cd local-data-warehouse/customer_satisfaction_in_airline/ && kaggle datasets download yakhyojon/customer-satisfaction-in-airline && cd ../../ && unzip local-data-warehouse/customer_satisfaction_in_airline/customer-satisfaction-in-airline.zip -d local-data-warehouse/customer_satisfaction_in_airline/ && rm local-data-warehouse/customer_satisfaction_in_airline/customer-satisfaction-in-airline.zip
""",
    # References
    academic_reference_bibtex="""@misc{yakhyojon2023airlinesatisfaction,
    author = {Kaggle User Yakhyojon},
    title = {Customer Satisfaction in Airline.},
    year = {2023},
    howpublished = {url{https://www.kaggle.com/datasets/yakhyojon/customer-satisfaction-in-airline}},
    note = {Kaggle}
}
""",
    academic_reference_bibtex_key="yakhyojon2023airlinesatisfaction",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We renamed the target column from "satisfaction" to "satisfied" for clarity and mapped the values to "Yes" and "No"
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="satisfied",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="satisfied",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/Invistico_Airline.csv", header=0)

feature_names = [
    "satisfaction",
    "Customer Type",
    "Age",
    "Type of Travel",
    "Class",
    "Flight Distance",
    "Seat comfort",
    "Departure/Arrival time convenient",
    "Food and drink",
    "Gate location",
    "Inflight wifi service",
    "Inflight entertainment",
    "Online support",
    "Ease of Online booking",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
    "Online boarding",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

df.rename(columns={"satisfaction": "satisfied"}, inplace=True)

cat_features = [
    "satisfied",
    "Customer Type",
    "Type of Travel",
    "Class",
    "Seat comfort",
    "Departure/Arrival time convenient",
    "Food and drink",
    "Gate location",
    "Inflight wifi service",
    "Inflight entertainment",
    "Online support",
    "Ease of Online booking",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
    "Online boarding",
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df["satisfied"] = df["satisfied"].map({"satisfied": "Yes", "dissatisfied": "No"})

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()